In [2]:
# 1. Imports
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import torch
import pandas as pd
import OpenAttack as oa
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, get_scheduler
from datasets import Dataset, DatasetDict
from OpenAttack.tags import TAG_English


c:\Users\Priyanka\AppData\Local\Programs\Python\Python38\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
MODEL_NM = "l3cube-pune/hindi-bert-v2"
DATA_PATH = "dataset-merged.csv"  # Ensure this file is in your directory
TEXT_COL = "text"               # Change to your CSV's text column name
LABEL_COL = "label"             # Change to your CSV's label column name
BATCH_SIZE = 8
EPOCHS = 3
MAX_LEN = 128

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Rows before cleaning: {len(df)}")
df = df.dropna(subset=[TEXT_COL])
df[TEXT_COL] = df[TEXT_COL].astype(str)
df = df[df[TEXT_COL].str.strip() != ""]
print(f"Rows after cleaning: {len(df)}")

if df[LABEL_COL].dtype == object:
    df[LABEL_COL] = pd.factorize(df[LABEL_COL])[0]

from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

raw_ds = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True))
})

Rows before cleaning: 17124
Rows after cleaning: 17123


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NM)

def tokenize_fn(batch):
    return tokenizer(batch[TEXT_COL], padding="max_length", truncation=True, max_length=MAX_LEN)

tokenized_ds = raw_ds.map(tokenize_fn, batched=True)
tokenized_ds = tokenized_ds.rename_column(LABEL_COL, "labels")
tokenized_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

train_loader = DataLoader(tokenized_ds["train"], shuffle=True, batch_size=BATCH_SIZE)
test_loader = DataLoader(tokenized_ds["test"], batch_size=BATCH_SIZE)

Map: 100%|██████████| 3425/3425 [00:00<00:00, 3511.54 examples/s]


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NM, num_labels=2).to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)
num_training_steps = EPOCHS * len(train_loader)
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)
model.to(device)

print("\n--- Starting Manual Training ---")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()

    model.eval()
    correct, total = 0, 0
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            logits = model(**batch).logits
        preds = torch.argmax(logits, dim=-1)
        correct += (preds == batch["labels"]).sum().item()
        total += batch["labels"].size(0)
    print(f"Epoch {epoch+1} Accuracy: {100 * correct / total:.2f}% | Avg Loss: {total_loss/len(train_loader):.4f}")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at l3cube-pune/hindi-bert-v2 and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\Priyanka\AppData\Local\Programs\Python\Python38\lib\site-packages\transformers\optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(



--- Starting Manual Training ---


Epoch 1:   0%|          | 0/1713 [00:00<?, ?it/s]c:\Users\Priyanka\AppData\Local\Programs\Python\Python38\lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
Epoch 1: 100%|██████████| 1713/1713 [16:16<00:00,  1.76it/s]


Epoch 1 Accuracy: 90.39% | Avg Loss: 0.3644


Epoch 2: 100%|██████████| 1713/1713 [15:11<00:00,  1.88it/s]


Epoch 2 Accuracy: 90.95% | Avg Loss: 0.2017


Epoch 3: 100%|██████████| 1713/1713 [15:02<00:00,  1.90it/s]


Epoch 3 Accuracy: 91.68% | Avg Loss: 0.1316


In [32]:
model.save_pretrained("model")
tokenizer.save_pretrained("model")

('model\\tokenizer_config.json',
 'model\\special_tokens_map.json',
 'model\\vocab.txt',
 'model\\added_tokens.json',
 'model\\tokenizer.json')

In [6]:
model = AutoModelForSequenceClassification.from_pretrained("model")
tokenizer = AutoTokenizer.from_pretrained("model")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(197285, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1

In [ ]:
# OpenAttack Evaluation
print("\n--- Starting Adversarial Attack Evaluation ---")
model.eval()

class SimpleTokenizer:
    TAGS = {TAG_English}

    def tokenize(self, text, pos_tagging=False):
        return text.split()

    def detokenize(self, tokens):
        return " ".join(tokens)

victim = oa.classifiers.TransformersClassifier(model, tokenizer, model.bert.embeddings.word_embeddings)
oa_dataset = raw_ds["test"].map(lambda x: {"x": x[TEXT_COL], "y": x[LABEL_COL]})
attacker = oa.attackers.DeepWordBugAttacker(tokenizer=SimpleTokenizer())
attack_eval = oa.AttackEval(attacker, victim)
results = attack_eval.eval(oa_dataset.select(range(50)), visualize=True)


--- Starting Adversarial Attack Evaluation ---


Map: 100%|██████████| 3425/3425 [00:00<00:00, 8186.63 examples/s]


Sample: 1 =====================================================================
Label: 0 (99.31%) --> Failed!               |                                   
                                            | Running Time:            0.0009992
पहल ट ी- 20 म ै च 44 रन प ा र ख े ल क े     | Query Exceeded:          no       
कनकशन न ि यम क े ट ी म ब ै ठ व ा ल जड े ज ब | Victim Model Queries:    21       
ा क ़ बच ट ी- 20 म ै च                      | Succeed:                 no       
                                            |                                   
Sample: 2 =====================================================================
Label: 0 (99.07%) --> Failed!               |                                   
                                            | Running Time:            0.0063338
RT ज ाँ च ह ो रह ी थ ी स ु श ां त क ी म ौ त | Query Exceeded:          no       
क ी और “ ग ि रफ ़् त ा र ी” ह ो रह ी ह ै “  | Victim Model Queries:    21       
ह ु क ् क ा” प ी न े क े ज ु र

Token indices sequence length is longer than the specified maximum sequence length for this model (599 > 512). Running this sequence through the model will result in indexing errors


Sample: 11 ====================================================================
Label: 0 (99.32%) --> Failed!               |                                   
                                            | Running Time:            0        
पर हर र ो ज स ु बह द े ख ें Zee आध ् य ा त  | Query Exceeded:          no       
् म ज ि सस े आपक े द ि न म ें भक ् त ि और   | Victim Model Queries:    18       
शक ् त ि द ो न ों रह े                      | Succeed:                 no       
                                            |                                   
Sample: 12 ====================================================================
Label: 1 (99.62%) --> Failed!               |                                   
                                            |                                   
ईम े ल एक च ौं क ा न े व ा ल ी नई र ि प ो र |                                   
् ट क े अन ु स ा र आईएसआईएस क े बर ् बर ल ो |                                   
ग ों न े 250 बच ् च ों क ो म ा

In [19]:
print("Attack Success Rate:", results["Attack Success Rate"])
print("Average Queries:", results["Avg. Victim Model Queries"])
print("Modification Rate: Not defined")

Attack Success Rate: 0.0
Average Queries: 124.04
Modification Rate: Not defined


In [8]:
from sklearn.metrics import accuracy_score

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def evaluate(model, dataloader):
    preds, labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            y = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            p = torch.argmax(outputs.logits, dim=1)

            preds.extend(p.cpu().numpy())
            labels.extend(y.cpu().numpy())

    return accuracy_score(labels, preds)

clean_acc = evaluate(model, test_loader)
print("Clean Accuracy:", clean_acc)

c:\Users\Priyanka\AppData\Local\Programs\Python\Python38\lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Clean Accuracy: 0.9167883211678832
